# 20. Grid-Search Model Selection

Per-animal BE vs SC model assignment using grid-search CV
on expert uniform sessions.


In [ ]:
%matplotlib inline
from shared_setup import *
import pickle
from pathlib import Path


In [ ]:
# ── Mode ──────────────────────────────────────────────────────────────────
# 'load' = load cluster results from results/
# 'run'  = quick local execution (small settings)
MODE = 'load'

if MODE == 'run':
    N_SYNTHETIC = 3
    N_GS_SEEDS = 2
    BURN_IN = 500
    N_SBI_SIMS = 1_000
    N_CV_REPEATS = 4
    SEED = 42
    print('MODE: run (quick local)')
else:
    print('MODE: load (cluster results)')


## 1. Load / Run Grid-Search Results

In [ ]:
RESULTS_DIR = Path('../results/cv')

gs_results = {}  # {fit_target: DataFrame}

if MODE == 'load':
    for ft in ['update_matrix', 'conditional_psych']:
        path = RESULTS_DIR / f'uniform_{ft}' / 'summary.pkl'
        if path.exists():
            with open(path, 'rb') as f:
                gs_results[ft] = pickle.load(f)['df']
            print(f'Loaded {ft}: {len(gs_results[ft])} animals')
        else:
            print(f'Missing: {path}')
            print('  Run: sbatch slurm/real_gs_uniform.sh')
            print('  Then: python scripts/gather_cv_results.py --distribution uniform --fit-target', ft)

elif MODE == 'run':
    from behav_utils.data.loading import list_animals, load_animal
    from behav_utils.data.selection import select_sessions
    from analysis.grid_search import grid_search_cv, DEFAULT_GRID

    animal_ids = list_animals()[:N_SYNTHETIC]  # limit for quick mode
    for ft in ['update_matrix', 'conditional_psych']:
        rows = []
        for aid in animal_ids:
            animal = load_animal(aid)
            sessions = select_sessions(animal, 'expert_uniform')
            if not sessions:
                continue
            be_errors, sc_errors = [], []
            for seed in range(1, N_GS_SEEDS + 1):
                for mt in ['BE', 'SC']:
                    try:
                        r = grid_search_cv(sessions, mt, seed=seed,
                                           burn_in=BURN_IN, fit_target=ft)
                        (be_errors if mt == 'BE' else sc_errors).append(r['avg_test_error'])
                    except Exception:
                        pass
            be_m = np.mean(be_errors) if be_errors else np.nan
            sc_m = np.mean(sc_errors) if sc_errors else np.nan
            rows.append({'animal': aid, 'BE': be_m, 'SC': sc_m,
                         'winner': 'BE' if be_m < sc_m else 'SC',
                         'n_sessions': len(sessions)})
        gs_results[ft] = pd.DataFrame(rows)
        print(f'{ft}: {len(rows)} animals')


## 2. Per-Animal Model Assignment

In [ ]:
for ft, df in gs_results.items():
    if len(df) == 0:
        continue
    print(f'\n=== {ft} ===')
    print(df.to_string(index=False, float_format='%.5f'))
    if 'winner' in df.columns:
        print(f'\nAssignment: {df["winner"].value_counts().to_dict()}')


## 3. Error Comparison (BE vs SC per Animal)

In [ ]:
for ft, df in gs_results.items():
    if len(df) == 0 or 'BE' not in df.columns:
        continue

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df))
    w = 0.35
    ax.bar(x - w/2, df['BE'], w, label='BE', color='steelblue', alpha=0.7)
    ax.bar(x + w/2, df['SC'], w, label='SC', color='darkorange', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(df['animal'], rotation=45, ha='right')
    ax.set_ylabel('Mean CV Error (MSE)')
    ax.set_title(f'GS Model Selection — {ft}')
    ax.legend()
    plt.tight_layout()
    plt.show()


## 4. Fit Target Comparison

Do update_matrix and conditional_psych agree on model assignments?


In [ ]:
if len(gs_results) == 2:
    um = gs_results.get('update_matrix', pd.DataFrame())
    cp = gs_results.get('conditional_psych', pd.DataFrame())
    if len(um) > 0 and len(cp) > 0:
        merged = um[['animal', 'winner']].rename(columns={'winner': 'um_winner'}).merge(
            cp[['animal', 'winner']].rename(columns={'winner': 'cp_winner'}),
            on='animal', how='inner',
        )
        merged['agree'] = merged['um_winner'] == merged['cp_winner']
        print(merged.to_string(index=False))
        print(f'\nAgreement: {merged["agree"].mean():.0%}')
    else:
        print('Need both fit targets to compare.')
else:
    print('Only one fit target available.')
